# Phase 1C - Table-only RAG (Colab runner)

Runner only: mount, git pull, install, call scripts. No logic lives here
(see CLAUDE.md P1/P2). Edit `src/` and `scripts/` locally, push, then re-run.

Table-only RAG over the Phase 1A/1B tables: serialize -> chunk -> retrieve -> grounded
answer. GT-filled and OCR-filled corpora stay strictly separate (P4); gold answers come
from GT, and the eval runs each corpus independently to measure the OCR-vs-GT QA gap.

The pipeline below: data prep (gt_filled headers, ocr_filled header backfill, corpora, QA
set) and BM25 retrieval eval are CPU / no key; dense + RRF retrieval and the LLM QA eval
are the GPU/API steps - the BGE embedding (dense) needs the GPU, and answer generation needs
an OpenRouter key (Step 5; default model `openai/gpt-4o-mini`).

## Boot

In [ ]:
# 1. Mount Drive (chunks / QA sets / rag index persist here).
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Get the code onto the VM, then pin the ACTIVE dev branch.
#    On a cold T4 VM /content/FinDocStructRAG does not exist yet, so clone it first; on a warm
#    VM (or a re-run) it is already there, so just update. `git pull` alone only updates the
#    CURRENT branch, so the branch is checked out explicitly and the HEAD hash is printed to
#    confirm the right code is live (a `skipped=300` in Step 0 means the wrong branch).
import os
BRANCH = 'feature/phase1c-retrieval'   # set to 'main' once Phase 1C is merged
if not os.path.isdir('/content/FinDocStructRAG/.git'):
    !git clone --quiet https://github.com/AD2000X/FinDocStructRAG.git /content/FinDocStructRAG
!cd /content/FinDocStructRAG && git fetch origin --quiet && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}
!cd /content/FinDocStructRAG && echo 'branch:' $(git rev-parse --abbrev-ref HEAD) ' HEAD:' $(git log --oneline -1)
%cd /content/FinDocStructRAG

In [ ]:
# 3. Make src/ importable and sanity-check the Phase 1C paths.
import sys, importlib
sys.path.insert(0, '/content/FinDocStructRAG')
from src import config
importlib.reload(config)  # refresh if the kernel imported an older config before the pull
print('IN_COLAB :', config.IN_COLAB)
print('CHUNKS   :', config.CHUNKS)
print('QA_DIR   :', config.QA_DIR)
print('QA seed  :', config.QA_MANUAL_SEED, '(exists:', config.QA_MANUAL_SEED.exists(), ')')

## Step 0 - (re)build gt_filled WITH headers (CPU)

Header detection (is_header) was added to `normalize_tatr_prediction`, but the gt_filled
tables on Drive were built before it. `--force` reprocesses all samples (ignoring the
resume manifest) so the regenerated tables carry column headers - which the linearized
serialization and the templated QA generator both need. Expect `processed=300 skipped=0`
(if you see `skipped=300`, --force is not in the checked-out code - fix the branch above).

In [ ]:
!python scripts/run_phase1b_gt_filled.py --limit 300 --seed 42 --run-id mvp_rand --force

In [ ]:
# Header sanity + FAIL-FAST. If it stops, the two prints say which cause:
#   (b) column_headers = 0  -> the GT annotation lacks header boxes (parse/class-name issue)
#   (a) column_headers > 0 but 0 header cells -> the marking overlap missed
import json, importlib
from src import config, fintabnet_loader as fl
importlib.reload(fl)

xmls = fl.find_xml_files(fl.structure_root(), limit=1, seed=42)
p = fl.parse_structure_xml(xmls[0])
print('sample class_counts   :', p['class_counts'])
print('sample column_headers :', len(p['column_headers']), 'boxes')

sample = sorted(config.TABLES_GT_FILLED.glob('*.json'))[:30]
with_headers = sum(1 for q in sample
                   if any(c.get('is_header') for c in json.loads(q.read_text())['cells']))
print(f'gt_filled (first {len(sample)}): {with_headers} have header cells')

assert with_headers > 0, (
    'No header cells marked - stopping so an empty QA set is not produced. See the two '
    'prints above for the cause (no header boxes vs marking missed).')

## Step 0b - backfill ocr_filled headers (CPU, no OCR re-run)

ocr_filled was built from tatr_predicted, whose grid carries no header marking, so its
cells are all `is_header=False`. gt_filled now has headers, so a GT-vs-OCR linearized
comparison would be unfair: gt_linearized pairs each value with its header, ocr_linearized
cannot. This reads the `column_headers` boxes from `tatr_raw/<id>.json` (same TATR
coordinate space) and marks `is_header` with the same IoMin rule - a fairness fix, no OCR
re-run. Expect most tables to report `>=1 hdr`.

In [ ]:
!python scripts/mark_ocr_filled_headers.py

## Step 1 - build retrieval corpora (CPU)

Serializes every gt_filled / ocr_filled table into one chunk per table and writes a JSONL
corpus per (text_source, serialization) under `rag_index/chunks/`. GT and OCR corpora are
kept separate (P4). With headers present, the linearized corpus now pairs each value with
its column header.

In [ ]:
!python scripts/build_table_chunks.py

## Step 2 - QA dataset (templated + manual seed) (CPU)

Generates ~30 templated lookup/numeric questions from GT cells (answer = GT cell) and
merges the hand-authored seed `qa/qa_manual_seed.jsonl` if present. With headers fixed,
`templated` should be ~30. Re-run after you author + push the manual set.

In [ ]:
!python scripts/build_qa_dataset.py --limit 30 --seed 42

## Step 3 - retrieval evaluation (CPU, no LLM)

BM25 over each corpus ({gt,ocr} x {markdown,linearized}) with the QA set; reports
hit@k / recall@k / MRR@k so markdown-vs-linearized and GT-vs-OCR are directly comparable.
No API key (P5); answer generation is a later step. Writes
`outputs/evaluation/rag/phase1c_retrieval.json`.

In [ ]:
!python scripts/evaluate_rag.py

## Step 3b - dense + RRF retrieval (GPU embedding)

Adds the dense path: BGE (`bge-small-en-v1.5`) embeddings of the chunks, exact cosine
search, behind the same `query -> ranked chunk_ids` contract as BM25, and `rrf` = RRF fusion
of BM25 + dense. This is the one piece that needs the GPU (the BGE forward pass); the search
itself is CPU. Runs all three methods so BM25 / dense / RRF print side by side per corpus.
Two hypotheses to check: does dense fix markdown's one hard miss (hit@10 < 1.0), and does
RRF get markdown's hit@1 together with linearized's recall@5? Overwrites the Step 3 report.

In [ ]:
# sentence-transformers pulls BGE; torch is already on the Colab image.
!pip install -q sentence-transformers
!python scripts/evaluate_rag.py --methods bm25,dense,rrf

## Step 4 - browse tables for manual QA authoring (CPU)

Displays a seeded sample of GT-filled tables as PNG images with each `sample_id` /
`chunk_id`, so you can read off true answers and write the manual + unanswerable
questions. Change `--limit` / `--seed` to see different tables. Use `%run` rather than
`!python` so IPython can render the images inline.

In [ ]:
%run scripts/preview_chunks.py --limit 25 --seed 7 --format image --display

## Step 5 - QA answer evaluation (GPU embedding + OpenRouter key)

The full RAG loop: RRF retrieval (BM25 + dense, the method carried forward) -> top-k chunks
-> LLM grounded answer (`src/llm_client`, OpenRouter / `openai/gpt-4o-mini`, single provider
P5). Scores answer_exact / numeric_relaxed / citation_hit (+ abstain_rate) per config
({gt,ocr} x {markdown,linearized}), GT/OCR separate (P4). top-k defaults to 10 because RRF
hit@10 is the only all-1.000 column. Needs an OpenRouter key (`OPENROUTER_API_KEY`; the cell
prompts for it with getpass if it is not already in the env). With only the 30 templated
lookups this validates the answer plumbing and the OCR-vs-GT answer gap; abstain_accuracy
fills in once the manual unanswerable questions are authored. Writes
`outputs/evaluation/rag/phase1c_qa.json` and per-config `outputs/qa/answers_<config>.jsonl`.

The cell prints the LLM-call count up front (4 corpora x 30 = 120 calls). `gpt-4o-mini` is
cheap pay-as-you-go and not rate-capped like a free tier, so the full run finishes quickly;
the completer still retries on a 429 (relevant if you swap in a `:free` model via
`OPENROUTER_MODEL`). Test the key cheaply first with
`scripts/evaluate_qa.py --limit 3 --corpus gt_markdown` (3 calls).

In [ ]:
# Step 5 setup (run ONCE per runtime): OpenRouter key + deps.
# The key goes into the kernel's os.environ, which propagates to the !python child process in
# BOTH the smoke and the full-run cells below - so set it here, not inside the run cell. That
# is why running a bare `!python scripts/evaluate_qa.py ...` before this cell fails with
# "set OPENROUTER_API_KEY". getpass works in the VS Code Colab setup where Colab Secrets
# (userdata) is not readable; the key is never written into the notebook.
import os
from getpass import getpass

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Paste OPENROUTER_API_KEY for this runtime: ")
os.environ["OPENROUTER_MODEL"] = "openai/gpt-4o-mini"
assert os.environ.get("OPENROUTER_API_KEY"), "Missing OPENROUTER_API_KEY"

!pip install -q sentence-transformers openai

In [ ]:
# Run the Step 5 setup cell above first (sets the key in the env this cell inherits).
# Smoke-test the key cheaply before the full run (5 calls); uncomment, run, then re-comment:
# !python scripts/evaluate_qa.py --limit 5 --corpus gt_linearized

# Full matrix: 4 corpora x 46 questions = 184 LLM calls.
!python scripts/evaluate_qa.py

## Author the manual set, then re-run Steps 2-5

The ~30 templated questions are automatic; the harder + unanswerable ones are hand-authored:

1. From the Step 4 output, pick ~10 tables for comparison/reasoning questions and a few for
   plausible-but-unanswerable ones.
2. **Locally** in VS Code, add one JSON object per line to `qa/qa_manual_seed.jsonl`
   (schema + examples in `qa/README.md`), using a real `sample_id` you just read.
3. `git push` -> re-run Boot, then **Step 2** (merge), **Step 3/3b** (re-score retrieval)
   and **Step 5** (re-score QA, now including abstain_accuracy on the unanswerable ones).

Dense + RRF retrieval (Step 3b) and LLM answer generation (Step 5) are in place; the manual
set is what turns Step 5 from a templated-lookup plumbing check into a full RAG measurement.